In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
office_table = dbutils.widgets.get("office_table")
client_table = dbutils.widgets.get("client_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW charges_src AS
SELECT 
  CAST(ReportingDate AS DATE) AS ReportingDate,
  NULL AS ChargeID,
  CAST(FacilityCode AS INT) AS FacilityCode,
  CAST(AcctNbr AS STRING) AS AcctNbr,
  NULL AS SvcDate,
  NULL AS PostDate,
  NULL AS ChgCode,
  NULL AS ChgDesc,
  NULL AS ChgDept,
  NULL AS ChgDeptDesc,
  NULL AS RevCode,
  NULL AS CPTCode,
  NULL AS Mod1,
  NULL AS Mod2,
  NULL AS Mod3,
  NULL AS Mod4,
  CAST(ChgQty AS INT) AS ChgQty,
  CAST(UnitAmt AS INT) AS UnitAmt,
  CAST(ChgAmt AS DOUBLE) AS ChgAmt,
  CAST(FinClass AS STRING) AS FinClass,
  NULL AS PrimaryPayorCode,
  NULL AS PrimaryPayorDesc,
  NULL AS PatientType,
  NULL AS AuthNbr,
  SourceSystemKey AS SourceSystemKey,
  current_timestamp() AS _load_timestamp
FROM (
  WITH 
  charges_cte AS (
    SELECT
      to_date(CAST(date_entered_key AS STRING), 'yyyyMMdd') AS ReportingDate,   
      ofc.OfficeNumber AS FacilityCode,
      CASE
        WHEN UPPER(obd.invoice_number) = 'ADV'
          THEN CONCAT('ADV - ', clt.SourceSystemId)
        ELSE obd.invoice_number
      END AS AcctNbr,
      obd.payor_type_code AS FinClass,
      0 AS ChgQty,
      0 AS UnitAmt,
      0 AS ChgAmt,
      0 AS SourceSystemKey
    FROM {source_table} obd
    LEFT JOIN {office_table} ofc
      ON ofc.OfficeKey = obd.office_key
    LEFT JOIN {client_table}  clt
      ON clt.ClientKey = obd.client_key
    WHERE obd.account_balance != 0
    AND obd.date_entered_key = date_format(DATE('{fetch_date}'), 'yyyyMMdd')
  ),
  charges_clean AS (
    SELECT *,
    row_number() OVER (
      PARTITION BY AcctNbr, FacilityCode 
      ORDER BY AcctNbr
    ) AS rn
    FROM charges_cte
  )
  SELECT ReportingDate, FacilityCode, AcctNbr, FinClass, ChgQty, UnitAmt, ChgAmt, SourceSystemKey
  FROM charges_clean
  WHERE rn=1
) AS src
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING charges_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 0

WHEN MATCHED THEN
UPDATE SET
    tgt.ChargeID = src.ChargeID,
    tgt.FacilityCode = src.FacilityCode,
    tgt.SvcDate = src.SvcDate,
    tgt.PostDate = src.PostDate,
    tgt.ChgCode = src.ChgCode,
    tgt.ChgDesc = src.ChgDesc,
    tgt.ChgDept = src.ChgDept,
    tgt.ChgDeptDesc = src.ChgDeptDesc,
    tgt.RevCode = src.RevCode,
    tgt.CPTCode = src.CPTCode,
    tgt.Mod1 = src.Mod1,
    tgt.Mod2 = src.Mod2,
    tgt.Mod3 = src.Mod3,
    tgt.Mod4 = src.Mod4,
    tgt.ChgQty = src.ChgQty,
    tgt.UnitAmt = src.UnitAmt,
    tgt.ChgAmt = src.ChgAmt,
    tgt.FinClass = src.FinClass,
    tgt.PrimaryPayorCode = src.PrimaryPayorCode,
    tgt.PrimaryPayorDesc = src.PrimaryPayorDesc,
    tgt.PatientType = src.PatientType,
    tgt.AuthNbr = src.AuthNbr,
    tgt.SourceSystemKey = src.SourceSystemKey,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    ChargeID,
    FacilityCode,
    AcctNbr,
    SvcDate,
    PostDate,
    ChgCode,
    ChgDesc,
    ChgDept,
    ChgDeptDesc,
    RevCode,
    CPTCode,
    Mod1,
    Mod2,
    Mod3,
    Mod4,
    ChgQty,
    UnitAmt,
    ChgAmt,
    FinClass,
    PrimaryPayorCode,
    PrimaryPayorDesc,
    PatientType,
    AuthNbr,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.ChargeID,
    src.FacilityCode,
    src.AcctNbr,
    src.SvcDate,
    src.PostDate,
    src.ChgCode,
    src.ChgDesc,
    src.ChgDept,
    src.ChgDeptDesc,
    src.RevCode,
    src.CPTCode,
    src.Mod1,
    src.Mod2,
    src.Mod3,
    src.Mod4,
    src.ChgQty,
    src.UnitAmt,
    src.ChgAmt,
    src.FinClass,
    src.PrimaryPayorCode,
    src.PrimaryPayorDesc,
    src.PatientType,
    src.AuthNbr,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)